## M: -1  环境与可行性

In [1]:
import cutlass
print(cutlass.__version__)

4.8.1


RTX 5070 Ti ×2, sm_120, 70 SMs, smem/block optin = 101376 B (~99 KB)
torch 2.11.0+cu130, nvcc 12.8, driver 580.178.04
nvidia-cutlass-dsl 4.8.1

## M0：朴素 attention + autograd.Function 管道验证
（迷你脚本对 SDPA验证前向+梯度，先隔离 plumbing 错误）← M1 前置

### step1 naive

```bash
forward:  S = Q@K^T * scale          # (B, T, T)
          mask (causal 时)
          P = softmax(S)
          O = P@V
          L = logsumexp(S)
保存: O, L, (Q, K, V 由 autograd 传入 backward)

backward: D_i = rowsum(dO ⊙ O)        # (B, T)，即 einsum('...qd,...qd->...q', dO, O)
          dP = dO @ V^T               # (B, T, T)
          dS = P * (dP - D[..., None])
          dV = P^T @ dO
          dQ = (dS @ K) * scale
          dK = (dS^T @ Q) * scale     # 注意两处 scale：S = QKᵀ·scale 求导各摊一半
```

In [3]:
"""M0：朴素（非 tiled）attention，作为 autograd.Function 协议与数学的第一道金标准。

与 M1/M2 的关系：
  - M0 保存 P（N×N），backward 直接用 P —— 数学最显然，gradcheck 靠它打底；
  - M1/M2 不保存 P，用 L 重算 —— 二者必须与 M0 的梯度一致（1e-4 容差）。
"""
import math

import torch
from einops import einsum
from jaxtyping import Bool, Float
from torch import Tensor


def naive_attention(
    Q: Float[Tensor, "batch seq d"],
    K: Float[Tensor, "batch seq d"],
    V: Float[Tensor, "batch seq d"],
    is_causal: bool = False,
) -> tuple[Float[Tensor, "batch seq d"], Float[Tensor, "batch seq seq"]]:
    """朴素前向：物化全部 N×N 中间量。

    返回:
      O: 与输入同 dtype
      P: softmax 概率 (B, N, N) —— 仅 M0 使用，M1 起禁止保存
    """
    d = Q.shape[-1]
    scale = 1.0 / math.sqrt(d)

    # S[b, i, j] = scale · Σ_d Q[b,i,d] · K[b,j,d] —— K 的行标 j 用 k，转置藏在下标里
    S: Float[Tensor, "batch seq seq"] = scale * einsum(Q, K, "b i d, b k d -> b i k")
    if is_causal:
        N = Q.shape[-2]
        mask: Bool[Tensor, "seq seq"] = torch.triu(
            torch.ones(N, N, dtype=torch.bool, device=Q.device), diagonal=1
        )
        S = S.masked_fill(mask, float("-inf"))

    P = torch.softmax(S, dim=-1)                    # (B, N, N)
    O = einsum(P, V, "b i k, b k d -> b i d")
    return O, P


class NaiveAttentionFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V, is_causal=False):
        O, P = naive_attention(Q, K, V, is_causal)
        ctx.save_for_backward(Q, K, V, P)           # M0 特例：存 P；M1 起换成 (O, L)
        ctx.is_causal = is_causal
        return O

    @staticmethod
    def backward(ctx, dO):
        Q, K, V, P = ctx.saved_tensors
        scale = 1.0 / math.sqrt(Q.shape[-1])        # 从 Q 的 shape 重取，不依赖 ctx 上的标量

        dV = einsum(P, dO, "b i k, b k d -> b i d")         # Pᵀ dO：i 求和，k 留在行
        dP = einsum(dO, V, "b i k, b k d -> b i k")         # dO Vᵀ：d 求和，k 留在列
        dS = P * (dP - einsum(P, dP, "b i k, b i k -> b i 1"))   # softmax 雅可比；行内求和
        dQ = scale * einsum(dS, K, "b i k, b k d -> b i d")
        dK = scale * einsum(dS, Q, "b i k, b i d -> b k d")  # dSᵀ Q：i 求和，k 留在行

        return dQ, dK, dV, None                     # 与 forward 参数 (Q, K, V, is_causal) 一一对应


```
tests/test_flash_gradcheck.py — M0/M1/M2 的第一轮验收：gradcheck + SDPA 对比。

① gradcheck：小形状、float64 —— 抓 autograd 协议错误和数学错误
② float32 与 SDPA 对照：O 和 dQ/dK/dV，容差 1e-4
③ 参数化覆盖：causal 两态、batch > 1、N=100（非 2 的幂）

跑法：
  uv run pytest tests/test_flash_gradcheck.py -v
```


### M1：把 forward 改成 tiled online softmax


```
Bq, Bk = 16, 16 之类的小 tile（先小后大，便于排错）
for i in range(ceil(N / Bq)):                       # Q 的 tile
    Qi = Q[i*Bq : (i+1)*Bq]
    O_i = 0  (Bq, d)  float32      l_i = 0 (Bq)      m_i = -inf (Bq)
    for j in range(ceil(N / Bk)):                   # K/V 的 tile
        S  = Qi @ Kj^T * scale                      # (Bq, Bk)，float32
        S  = where(q_idx >= k_idx, S, -inf)         # 仅 causal
        m_new = max(m_i, rowmax(S))
        P  = exp(S - m_new[:, None])
        alpha = exp(m_i - m_new)                    # 旧结果的缩放因子
        l_i = alpha * l_i + rowsum(P)
        O_i = alpha[:, None] * O_i + P @ Vj
        m_i = m_new
    O[i] = O_i / l_i[:, None]
    L[i] = m_i + log(l_i)                           # 自然对数的 logsumexp（已含缩放）
```

In [ ]:
"""M1：纯 PyTorch 的 tiled FlashAttention-2 forward（jaxtyping + einops 版）。"""
import math

import torch
from einops import einsum, rearrange, reduce
from jaxtyping import Bool, Float, Int, Float32
from torch import Tensor


def flash_fwd_tiled(
    Q: Float[Tensor, "batch seq d"],
    K: Float[Tensor, "batch seq d"],
    V: Float[Tensor, "batch seq d"],
    is_causal: bool = False,
    Bq: int = 64,
    Bk: int = 64,
) -> tuple[Float[Tensor, "batch seq d"], Float[Tensor, "batch seq"]]:
    B, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    dev = Q.device
    O = torch.empty_like(Q)
    L = torch.empty(B, N, dtype=torch.float32, device=dev)

    for qs in range(0, N, Bq):
        qe = min(qs + Bq, N)
        Qt = rearrange(Q[:, qs:qe], "b tile d -> b tile d").float()   # 切片，语义标注用
        m   = torch.full((B, qe - qs), float("-inf"), device=dev)
        l   = torch.zeros((B, qe - qs), device=dev)
        acc = torch.zeros((B, qe - qs, d), device=dev)

        for ks in range(0, N, Bk):
            if is_causal and ks > qe - 1:
                break
            ke = min(ks + Bk, N)
            Kj = rearrange(K[:, ks:ke], "b k d -> b k d").float()
            Vj = rearrange(V[:, ks:ke], "b k d -> b k d").float()

            # einsum 用具名维度：q=查询行, k=键列, d=特征维
            S = scale * einsum(Qt, Kj, "b q d, b k d -> b q k")
            if is_causal:
                q_idx: Int[Tensor, "q 1"] = torch.arange(qs, qe, device=dev)[:, None]
                k_idx: Int[Tensor, "1 k"] = torch.arange(ks, ke, device=dev)[None, :]
                S = S.masked_fill(q_idx < k_idx, float("-inf"))

            m_new = reduce(S, "b q k -> b q", "max")           # 行最大值
            P     = torch.exp(S - rearrange(m_new, "b q -> b q 1"))
            alpha = torch.exp(m - m_new)
            l     = alpha * l + reduce(P, "b q k -> b q", "sum")
            acc   = rearrange(alpha, "b q -> b q 1") * acc + einsum(P, Vj, "b q k, b k d -> b q d")
            m     = m_new

        O[:, qs:qe] = rearrange(acc / rearrange(l, "b q -> b q 1"), "b q d -> b q d").to(Q.dtype)
        L[:, qs:qe] = m + torch.log(l)
    return O, L


### M2：tiled backward（重算 P，不重做 softmax）


- 两个关键恒等式

```
P_ij = exp(S_ij − L_i)                    # 用保存的 L 重算，不需要 m、l
D_i  = rowsum(dO ⊙ O)_i  =  Σ_j P_ij · dP_ij     # 化简，免去一次 O(N²) 归约
dS   = P ⊙ (dP − D[:, None])
```

- 算法骨架

```
D = (dO * O).sum(-1)                                      # (N)，先算好
for j in range(Tk):                                       # 外层：K/V tile
    dK_j = 0 ; dV_j = 0                                   # float32
    for i in range(Tq):                                   # 内层：Q tile（causal 时可跳过完全看不到 K_j 的 Q tile）
        S  = Qi @ Kj^T * scale ;  mask
        P  = exp(S − L_i[:, None])
        dV_j += P^T @ dO_i
        dP = dO_i @ Vj^T
        dS = P * (dP − D_i[:, None])
        dQ_i += dS @ Kj * scale                           # 跨 j 累加
        dK_j += dS^T @ Qi * scale
```

In [ ]:
def flash_bwd_tiled(
    Q: Float[Tensor, "batch seq d"],
    K: Float[Tensor, "batch seq d"],
    V: Float[Tensor, "batch seq d"],
    O: Float[Tensor, "batch seq d"],
    L: Float32[Tensor, "batch seq"],
    dO: Float[Tensor, "batch seq d"],
    is_causal: bool = False,
    Bq: int = 64,
    Bk: int = 64,
) -> tuple[
    Float[Tensor, "batch seq d"],
    Float[Tensor, "batch seq d"],
    Float[Tensor, "batch seq d"],
]:
    """Tiled FlashAttention-2 backward。

    输入：
      Q, K, V : (B, N, d)      原始输入（M1 forward 保存的，dtype 任意浮点）
      O       : (B, N, d)      M1 forward 的输出
      L       : (B, N) fp32    M1 forward 的 logsumexp（含 1/√d，自然对数）
      dO      : (B, N, d)      输出梯度
    输出：
      dQ, dK, dV : 各 (B, N, d)，dtype 与输入一致
    """
    B, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    dev = Q.device

    D = einsum(dO.float(), O.float(), "b q d, b q d -> b q")   # (B, N) fp32

    dQ = torch.zeros(B, N, d, dtype=torch.float32, device=dev)
    dK = torch.zeros(B, N, d, dtype=torch.float32, device=dev)
    dV = torch.zeros(B, N, d, dtype=torch.float32, device=dev)

    for qs in range(0, N, Bq):                          # 外层：Q tile
        qe = min(qs + Bq, N)
        Qt  = Q[:, qs:qe].float()
        dOt = dO[:, qs:qe].float()
        Lt  = L[:, qs:qe]
        Dt  = D[:, qs:qe]

        for ks in range(0, N, Bk):                      # 内层：K/V tile
            if is_causal and ks >= qe:                  # k 列全部 > q 行：整块 mask，跳过
                break
            ke = min(ks + Bk, N)
            Kj = K[:, ks:ke].float()
            Vj = V[:, ks:ke].float()

            S = scale * einsum(Qt, Kj, "b q d, b k d -> b q k")
            if is_causal:                               # mask 用全局下标
                q_idx: Int[Tensor, "q 1"] = torch.arange(qs, qe, device=dev)[:, None]
                k_idx: Int[Tensor, "1 k"] = torch.arange(ks, ke, device=dev)[None, :]
                S = S.masked_fill(q_idx < k_idx, float("-inf"))

            P = torch.exp(S - rearrange(Lt, "b q -> b q 1"))     # 用保存的 L 重算 P

            dV[:, ks:ke] += einsum(P, dOt, "b q k, b q d -> b k d")
            dP = einsum(dOt, Vj, "b q d, b k d -> b q k")
            dS = P * (dP - rearrange(Dt, "b q -> b q 1"))
            dK[:, ks:ke] += scale * einsum(dS, Qt, "b q k, b q d -> b k d")
            dQ[:, qs:qe] += scale * einsum(dS, Kj, "b q k, b k d -> b q d")

    return dQ.to(Q.dtype), dK.to(Q.dtype), dV.to(Q.dtype)


In [ ]:

class FlashAttnPytorch(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V, is_causal=False):
        O, L = flash_fwd_tiled(Q, K, V, is_causal)
        ctx.save_for_backward(Q, K, V, O, L)            # 只存这五个，不存 P
        ctx.is_causal = is_causal
        return O

    @staticmethod
    def backward(ctx, dO):
        Q, K, V, O, L = ctx.saved_tensors
        dQ, dK, dV = flash_bwd_tiled(Q, K, V, O, L, dO, ctx.is_causal)
        return dQ, dK, dV, None

In [ ]:
"""tests/test_flash_all.py — M0（朴素）/ M1（tiled forward）/ M2（tiled backward）完整验收。

覆盖：
  ① gradcheck（fp64，最强）            —— M0 和 M1+M2 各跑一遍
  ② 前向：M1 vs fp64 参考              —— online softmax、L 的定义
  ③ 反向：M2 vs fp64 参考              —— L 重算 P、D 恒等式、scale 位置
  ④ 三角互证：M0 vs M1+M2 的 O 与梯度  —— 两套独立实现必须一致
  ⑤ 契约：saved_tensors 恰好一个 (B,N) fp32，不存 N×N
  ⑥ tile 一致性：换 (Bq,Bk) 不改变任何输出（含 qs 外层的新循环顺序）
  ⑦ bf16：O 容差 1e-2，L/梯度 fp32 路径仍准
  ⑧ SDPA 交叉验证（GPU 可选）

跑法：
  uv run pytest tests/test_flash_all.py -v          # 全量
  uv run pytest tests/test_flash_all.py -v -m "not gpu"   # 跳过 GPU 测试
  uv run python tests/test_flash_all.py             # 快速自检子集
"""
import math

import pytest
import torch
from jaxtyping import Float
from torch import Tensor


ALL_IMPLS = pytest.mark.parametrize("impl", ["naive", "flash"])


# ================================================================ 工具

def reference(Q, K, V, causal: bool):
    """fp64 全量参考：返回 (Qf, Kf, Vf, O, L)，均带 requires_grad 以便反向。"""
    Qf = Q.detach().double().requires_grad_()
    Kf = K.detach().double().requires_grad_()
    Vf = V.detach().double().requires_grad_()
    S = (Qf @ Kf.transpose(-2, -1)) / math.sqrt(Q.shape[-1])
    if causal:
        N = Q.shape[1]
        S = S.masked_fill(
            torch.triu(torch.ones(N, N, dtype=torch.bool, device=Q.device), 1),
            float("-inf"),
        )
    L = torch.logsumexp(S, -1)
    O = torch.softmax(S, -1) @ Vf
    return Qf, Kf, Vf, O, L


def make_inputs(B, N, d, device="cpu", dtype=torch.float32, requires_grad=False, seed=0):
    torch.manual_seed(seed)
    gen = (torch.randn(B, N, d, device=device, dtype=dtype) for _ in range(3))
    return tuple(t.requires_grad_(requires_grad) for t in gen)


def run_forward(impl, Q, K, V, causal):
    """统一前向入口：返回 O（保留计算图）。"""
    fn = NaiveAttentionFn if impl == "naive" else FlashAttnPytorch
    return fn.apply(Q, K, V, causal)


def grads_of(*tensors):
    return [t.grad for t in tensors]


# ================================================================ ① gradcheck

@pytest.mark.parametrize("causal", [False, True])
@pytest.mark.parametrize("N,d", [(16, 8), (17, 4)])
def test_gradcheck_naive(causal, N, d):
    Q, K, V = make_inputs(2, N, d, dtype=torch.float64, requires_grad=True, seed=1)
    assert torch.autograd.gradcheck(
        NaiveAttentionFn.apply, (Q, K, V, causal), eps=1e-6, atol=1e-6, rtol=1e-4)


@pytest.mark.parametrize("causal", [False, True])
@pytest.mark.parametrize("N,d", [(16, 8), (17, 4)])
def test_gradcheck_flash(causal, N, d):
    Q, K, V = make_inputs(2, N, d, dtype=torch.float64, requires_grad=True, seed=1)
    assert torch.autograd.gradcheck(
        FlashAttnPytorch.apply, (Q, K, V, causal), eps=1e-6, atol=1e-6, rtol=1e-4)


# ================================================================ ② M1 前向 vs fp64 参考

@pytest.mark.parametrize("causal", [False, True])
@pytest.mark.parametrize("N", [1, 16, 17, 100, 128, 1000])
@pytest.mark.parametrize("d", [16, 64])
def test_m1_forward_vs_reference(causal, N, d):
    Q, K, V = make_inputs(2, N, d)
    _, _, _, O_ref, L_ref = reference(Q, K, V, causal)
    O, L = flash_fwd_tiled(Q, K, V, causal)
    torch.testing.assert_close(O, O_ref, atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(L, L_ref, atol=1e-5, rtol=1e-5)   # L 比 O 严一档


# ================================================================ ③ M2 反向 vs fp64 参考

@pytest.mark.parametrize("causal", [False, True])
@pytest.mark.parametrize("N", [1, 16, 17, 100, 128])
@pytest.mark.parametrize("d", [16, 64])
def test_m2_backward_vs_reference(causal, N, d):
    Q, K, V = make_inputs(2, N, d, requires_grad=True)
    Qf, Kf, Vf, O_ref, _ = reference(Q, K, V, causal)
    dO = torch.randn(2, N, d, generator=torch.Generator().manual_seed(1))

    O = FlashAttnPytorch.apply(Q, K, V, causal)
    O.backward(dO)
    O_ref.backward(dO.double())

    for g, ref, name in ((Q.grad, Qf.grad, "dQ"), (K.grad, Kf.grad, "dK"), (V.grad, Vf.grad, "dV")):
        assert g is not None, f"{name} 为 None"
        torch.testing.assert_close(
            g.double(), ref, atol=1e-5, rtol=1e-4, msg=lambda m, n=name: f"{n}: {m}")


# ================================================================ ④ 三角互证：M0 vs M1+M2

@pytest.mark.parametrize("causal", [False, True])
@pytest.mark.parametrize("B,N,d", [(2, 1, 16), (2, 16, 32), (1, 100, 32), (2, 128, 64)])
def test_m0_vs_m1_m2(causal, B, N, d):
    """两套独立实现（M0 存 P 直算 vs M1/M2 用 L 重算）的 O 与三个梯度必须一致。"""
    Q0, K0, V0 = make_inputs(B, N, d, requires_grad=True, seed=3)
    Q2, K2, V2 = make_inputs(B, N, d, requires_grad=True, seed=3)   # 同一批数据
    assert torch.equal(Q0, Q2) and torch.equal(V0, V2)
    dO = torch.randn(B, N, d, generator=torch.Generator().manual_seed(4))

    O_naive = run_forward("naive", Q0, K0, V0, causal)
    O_flash = run_forward("flash", Q2, K2, V2, causal)

    # 前向一致
    torch.testing.assert_close(O_naive, O_flash, atol=1e-5, rtol=1e-5)

    # 反向一致：用同一 dO，各自 backward
    O_naive.backward(dO)
    O_flash.backward(dO)
    for g0, g2, name in ((Q0.grad, Q2.grad, "dQ"), (K0.grad, K2.grad, "dK"), (V0.grad, V2.grad, "dV")):
        assert g0 is not None and g2 is not None, f"{name} 为 None"
        torch.testing.assert_close(g0, g2, atol=1e-4, rtol=1e-4, msg=lambda m, n=name: f"{n}: {m}")


# ================================================================ ⑤ 契约检查

def test_contract_saved_tensors():
    """M1 契约：恰好一个 (B,N) 的 fp32 张量（L），且不保存 N×N 矩阵。"""
    Q, K, V = make_inputs(2, 64, 16, requires_grad=True)
    O = FlashAttnPytorch.apply(Q, K, V, True)
    saved = O.grad_fn.saved_tensors

    lse = [t for t in saved if t.shape == Q.shape[:2]]
    assert len(lse) == 1 and lse[0].dtype == torch.float32, \
        f"应恰好保存一个 (B,N) fp32 的 L，实际 {[(tuple(t.shape), t.dtype) for t in saved]}"

    # L 的值必须是对数域 logsumexp（不是别的 (B,N) 张量）
    _, _, _, _, L_ref = reference(Q, K, V, True)
    torch.testing.assert_close(lse[0], L_ref.float(), atol=1e-5, rtol=1e-5)

    # 不保存 N×N 的 P/S：所有保存的张量都不该是 (…, N, N) 形状
    N = Q.shape[1]
    assert not any(t.dim() >= 2 and t.shape[-2] == N and t.shape[-1] == N for t in saved), \
        "疑似保存了 N×N 矩阵（P/S）——违反 FlashAttention 契约"


def test_contract_backward_protocol():
    """backward 返回值个数 = forward 参数个数（4），is_causal 位置为 None。"""
    Q, K, V = make_inputs(2, 32, 16, requires_grad=True)
    O = FlashAttnPytorch.apply(Q, K, V, False)
    O.backward(torch.randn_like(O))
    for g, t, name in ((Q.grad, Q, "dQ"), (K.grad, K, "dK"), (V.grad, V, "dV")):
        assert g.shape == t.shape and g.dtype == t.dtype, f"{name} 形状/dtype 不符"


# ================================================================ ⑥ tile 一致性

@pytest.mark.parametrize("causal", [False, True])
def test_tile_invariance_forward(causal):
    """换 (Bq,Bk)（含 Bq≠Bk、不整除）的 O/L 必须一致。"""
    B, N, d = 2, 100, 32
    Q, K, V = make_inputs(B, N, d, seed=5)

    base_O, base_L = flash_fwd_tiled(Q, K, V, causal, 16, 16)
    for Bq, Bk in [(64, 32), (7, 5), (128, 128)]:
        O, L = flash_fwd_tiled(Q, K, V, causal, Bq, Bk)
        torch.testing.assert_close(O, base_O, atol=1e-5, rtol=1e-5,
                                   msg=lambda m, t=(Bq, Bk): f"tile={t}: O 不一致 {m}")
        torch.testing.assert_close(L, base_L, atol=1e-5, rtol=1e-5,
                                   msg=lambda m, t=(Bq, Bk): f"tile={t}: L 不一致 {m}")


@pytest.mark.parametrize("causal", [False, True])
def test_tile_invariance_backward(causal):
    """换 (Bq,Bk) 的 dQ/dK/dV 必须一致——专测 M2 的 qs 外层新循环顺序。"""
    B, N, d = 2, 100, 32
    Q, K, V = make_inputs(B, N, d, seed=5)
    dO = torch.randn(B, N, d, generator=torch.Generator().manual_seed(6))

    base = flash_fwd_tiled(Q, K, V, causal, 16, 16)
    base_grads = flash_bwd_tiled(*base, dO, causal, 16, 16)
    for Bq, Bk in [(64, 32), (7, 5), (128, 128)]:
        grads = flash_bwd_tiled(*base, dO, causal, Bq, Bk)
        for a, b, name in zip(base_grads, grads, ("dQ", "dK", "dV")):
            torch.testing.assert_close(a, b, atol=1e-5, rtol=1e-5,
                                       msg=lambda m, t=(Bq, Bk), n=name: f"tile={t}: {n} {m}")


# ================================================================ ⑦ bf16

@pytest.mark.parametrize("causal", [False, True])
def test_bf16(causal):
    """内部 fp32：O 误差 ~1e-2（最后舍入到 bf16），L 仍应很准（1e-5）。"""
    B, N, d = 2, 1000, 64
    Q, K, V = make_inputs(B, N, d, dtype=torch.bfloat16, seed=7)
    Qf, Kf, Vf, O_ref, L_ref = reference(Q, K, V, causal)   # 参考也用舍入后的输入

    O, L = flash_fwd_tiled(Q, K, V, causal, 64, 64)
    eO = (O.double() - O_ref).abs().max().item()
    eL = (L.double() - L_ref).abs().max().item()
    assert eO < 2e-2, f"O 误差过大: {eO:.2e}"
    assert eL < 1e-5, f"L 误差过大: {eL:.2e}"

    # 反向路径（bf16 输入 → fp32 累加 → bf16 梯度），容差 2e-2
    Qg, Kg, Vg = (t.detach().clone().requires_grad_() for t in (Q, K, V))
    O_flash = FlashAttnPytorch.apply(Qg, Kg, Vg, causal)
    dO = torch.randn(B, N, d, dtype=torch.bfloat16, generator=torch.Generator().manual_seed(8))
    O_flash.backward(dO)
    O_ref.backward(dO.double())
    for g, ref, name in ((Qg.grad, Qf.grad, "dQ"), (Kg.grad, Kf.grad, "dK"), (Vg.grad, Vf.grad, "dV")):
        torch.testing.assert_close(
            g.double(), ref, atol=2e-2, rtol=2e-2, msg=lambda m, n=name: f"bf16 {n}: {m}")


# ================================================================ ⑧ SDPA 交叉验证（GPU）

@pytest.mark.gpu
@pytest.mark.parametrize("causal", [False, True])
def test_vs_sdpa(causal):
    if not torch.cuda.is_available():
        pytest.skip("需要 GPU")
    B, N, d = 2, 512, 64
    Q, K, V = make_inputs(B, N, d, "cuda", requires_grad=True, seed=9)
    Qf, Kf, Vf, _, _ = reference(Q, K, V, causal)
    dO = torch.randn(B, N, d, device="cuda")

    O = FlashAttnPytorch.apply(Q, K, V, causal)
    O.backward(dO)
    ref = torch.nn.functional.scaled_dot_product_attention(Qf, Kf, Vf, is_causal=causal)
    ref.backward(dO.double())
    torch.testing.assert_close(O, ref, atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(Q.grad, Qf.grad, atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(K.grad, Kf.grad, atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(V.grad, Vf.grad, atol=1e-4, rtol=1e-4)


# ================================================================ 直接运行入口（快速子集）


print("== 快速自检（完整版用 pytest 跑）==")
for causal in (False, True):
    Q, K, V = make_inputs(2, 16, 8, dtype=torch.float64, requires_grad=True, seed=1)
    assert torch.autograd.gradcheck(
        NaiveAttentionFn.apply, (Q, K, V, causal), eps=1e-6, atol=1e-6)
    assert torch.autograd.gradcheck(
        FlashAttnPytorch.apply, (Q, K, V, causal), eps=1e-6, atol=1e-6)
    print(f"gradcheck causal={causal} ✓")

# 三角互证
for causal in (False, True):
    Q0, K0, V0 = make_inputs(2, 100, 32, requires_grad=True, seed=3)
    Q2, K2, V2 = make_inputs(2, 100, 32, requires_grad=True, seed=3)
    dO = torch.randn(2, 100, 32, generator=torch.Generator().manual_seed(4))
    run_forward("naive", Q0, K0, V0, causal).backward(dO)
    run_forward("flash", Q2, K2, V2, causal).backward(dO)
    for a, b, n in ((Q0.grad, Q2.grad, "dQ"), (K0.grad, K2.grad, "dK"), (V0.grad, V2.grad, "dV")):
        torch.testing.assert_close(a, b, atol=1e-4, rtol=1e-4, msg=lambda m, x=n: f"{x}: {m}")
    print(f"M0 vs M1+M2 causal={causal} ✓")

test_contract_saved_tensors()
print("契约 ✓")
print("快速自检全部通过")


In [ ]:
run_checks()

- 见cute_dsl/flash_*

### M6：调参、benchmark-记录

#### tiles

(vllm) (cs336-assignment2) (base) c220@c220:/data2/c220/hz/cs336/cute_dsl$ uv run python sweep_tiles.py
sK layout: S<3,3,3> o 0 o (32,64,2):(64,1,2048)
sK layout: S<3,3,3> o 0 o (32,64,2):(64,1,2048)
sK layout: S<3,3,3> o 0 o (64,64,2):(64,1,4096)
sK layout: S<3,3,3> o 0 o (64,64,2):(64,1,4096)
sK layout: S<3,3,3> o 0 o (128,64,2):(64,1,8192)
sK layout: S<3,3,3> o 0 o (64,64,1):(64,1,4096)
sK layout: S<3,3,3> o 0 o (64,64,3):(64,1,4096)
sK layout: S<3,3,3> o 0 o (64,64,3):(64,1,4096)

SM 数 = 70；forward、causal、d=64、bf16；TFLOP/s = 2·N²·d / 时间

| Bq | Bk | stages | warps | smem(KB) | 编译(s) | 状态 | N=4096 blocks | ms | TFLOP/s | N=16384 blocks | ms | TFLOP/s |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 32 | 32 | 2 | 2 | 20 | 0.5 | OK | 128 (1.83x SM) | 0.086 | 24.9 | 512 (7.31x SM) | 0.587 | 58.5 |
| 64 | 32 | 2 | 4 | 24 | 0.4 | OK | 64 (0.91x SM) | 0.087 | 24.7 | 256 (3.66x SM) | 0.570 | 60.3 |
| 64 | 64 | 2 | 4 | 40 | 0.5 | OK | 64 (0.91x SM) | 0.076 | 28.4 | 256 (3.66x SM) | 0.555 | 61.9 |
| 128 | 64 | 2 | 8 | 48 | 0.5 | OK | 32 (0.46x SM) | 0.128 | 16.7 | 128 (1.83x SM) | 0.691 | 49.7 |
| 128 | 128 | 2 | 8 | 80 | 0.7 | OK | 32 (0.46x SM) | 0.121 | 17.7 | 128 (1.83x SM) | 0.646 | 53.2 |
| 64 | 64 | 1 | 4 | 24 | 0.5 | OK | 64 (0.91x SM) | 0.091 | 23.7 | 256 (3.66x SM) | 0.581 | 59.1 |
| 64 | 64 | 3 | 4 | 56 | 0.6 | OK | 64 (0.91x SM) | 0.079 | 27.3 | 256 (3.66x SM) | 0.639 | 53.8 |
| 128 | 64 | 3 | 8 | 64 | 0.5 | OK | 32 (0.46x SM) | 0.125 | 17.1 | 128 (1.83x SM) | 0.673 | 51.0 |

#### bench

(vllm) (cs336-assignment2) (base) c220@c220:/data2/c220/hz/cs336/cute_dsl$ uv run python bench_flash.py
sK layout: S<3,3,3> o 0 o (64,64,2):(64,1,4096)
[compile] FA fwd+bwd 首次调用（含 JIT）：1.9 s

## dtype=bf16：cuBLAS 4096³ 峰值 = 98.1 TFLOP/s（= 100%）

### d=64，causal=True，CuTe FA 支持：True

| N | naive fwd ms | SDPA fwd ms | FA fwd e2e ms | FA fwd kernel ms | FA fwd TFLOP/s | 利用率 | naive f+b ms | FA f+b ms | FA f+b TFLOP/s |
|---|---|---|---|---|---|---|---|---|---|
| 128 | 0.134 | 0.063 | 0.100 | 0.014 | 0.1 | 0.1% | 0.550 | 0.638 | 0.0 |
| 512 | 0.133 | 0.054 | 0.102 | 0.019 | 1.7 | 1.8% | 0.609 | 0.642 | 0.2 |
| 2048 | 0.145 | 0.055 | 0.114 | 0.044 | 12.3 | 12.5% | 0.659 | 0.790 | 2.4 |
| 8192 | 2.564 | 0.241 | 0.265 | 0.193 | 44.4 | 45.3% | 5.438 | 1.050 | 28.6 |
| 32768 | 39.702 | 1.917 | 1.957 | 1.908 | 72.0 | 73.4% | 86.240 | 8.388 | 57.3 |

SDPA 实际使用的后端：flash

显存（增量峰值 GiB / 绝对峰值 GiB）：

| N | naive fwd | naive f+b | FA fwd | FA f+b |
|---|---|---|---|---|
| 128 | 0.000 / 0.016 | 0.000 / 0.016 | 0.000 / 0.016 | 0.000 / 0.016 |
| 512 | 0.001 / 0.018 | 0.003 / 0.019 | 0.000 / 0.016 | 0.001 / 0.017 |
| 2048 | 0.023 / 0.041 | 0.036 / 0.053 | 0.000 / 0.017 | 0.002 / 0.020 |
| 8192 | 0.375 / 0.396 | 0.567 / 0.588 | 0.001 / 0.022 | 0.010 / 0.031 |
| 32768 | 6.000 / 6.035 | 9.020 / 9.055 | 0.004 / 0.040 | 0.039 / 0.075 |

log-log 斜率（增量峰值显存 vs N）：FA fwd = 1.0，naive fwd = 2.0（预测 FA≈1，基线≈2）

#### ncu

```bash
sudo env CUDA_VISIBLE_DEVICES=0 /usr/local/cuda/bin/ncu  --set full -k regex:FLASH_ATTN -c 1 -o fa_fwd   /data2/c220/hz/cs336/.venv/bin/python \
-c "import torch; from flash_attn_autograd import flash_attn_func as f; \
q,k,v=[torch.randn(1,16384,64,device='cuda',dtype=torch.bfloat16) for _ in range(3)]; \
[f(q,k,v,True) for _ in range(3)]; torch.cuda.synchronize()"
```

```bash
# a. 精确定位 f+b 的交叉点
uv run python bench_flash.py --seqs 1024 2048 3072 4096 6144 8192 --iters 50
# b. 单 kernel 反向，并补 65536（基线应 OOM，这就是你要的 OOM 点）
uv run python bench_flash.py --seqs 8192 32768 65536 --fused
# c. causal 与非 causal 对比（用来验证负载不均衡）
uv run python bench_flash.py --seqs 2048 8192 32768 --no-causal
# d. 另一个计时口径（清 L2），单独成表
uv run python bench_flash.py --seqs 2048 8192 32768 --flush-l2
# e. 把 ncu 报告的剩余部分打印出来（重点：Warp State Statistics、Occupancy、Launch Statistics）
ncu --import fa_fwd.ncu-rep --page details | sed -n '80,260p'
```